In [33]:
# Cell 1
!pip install -q tensorflow_decision_forests seaborn wurlitzer

In [34]:
# Cell 2
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_decision_forests as tfdf
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

print("TensorFlow:", tf.__version__)
print("TFDF:", tfdf.__version__)

TensorFlow: 2.19.0
TFDF: 1.12.0


In [35]:
# Cell 3
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
train.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [36]:
# Cell 4
train = train.drop("Id", axis=1)
test_ids = test["Id"]
test = test.drop("Id", axis=1)

In [37]:
# Cell 5
def split_fixed(df, test_ratio=0.2, seed=42):
    np.random.seed(seed)
    m = np.random.rand(len(df)) < test_ratio
    return df[~m], df[m]

In [38]:
# Cell 6: 定義特徵和目標
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)

# 初始化欄位清單
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("數值欄位數量 =", len(num_cols))
print("類別欄位數量 =", len(cat_cols))


數值欄位數量 = 36
類別欄位數量 = 43


In [39]:
# Cell 7: 特徵工程（資料整合）- 建立 TotalQualityIndex (調整權重)

def create_total_quality_index(df):
    # 再次調整權重：嘗試更平衡的分配，增加 GrLivArea 和 TotalBsmtSF 的相對影響。
    df["TotalQualityIndex"] = (
        df["OverallQual"] * 0.25 + # 權重從 0.5 調整為 0.25
        df["GrLivArea"] / 5000 * 0.40 + # 權重從 0.3 調整為 0.40
        df["TotalBsmtSF"].fillna(0) / 3000 * 0.25 + # 權重從 0.1 調整為 0.25
        df["GarageCars"].fillna(0) * 0.10 # 權重保持 0.10
    )
    return df

# 對訓練集和測試集應用特徵工程
X = create_total_quality_index(X.copy())
test = create_total_quality_index(test.copy())

# 重新計算欄位清單，以包含新的衍生特徵
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("✅ 已新增特徵：TotalQualityIndex (使用新權重)")
print("更新後數值欄位數量 =", len(num_cols))
print("更新後類別欄位數量 =", len(cat_cols))


✅ 已新增特徵：TotalQualityIndex (使用新權重)
更新後數值欄位數量 = 37
更新後類別欄位數量 = 43


In [40]:
# Cell 8: scikit-learn Pipeline 訓練 (用於計算特徵重要性)

from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 數值補中位數
num_tf = Pipeline([("imputer", SimpleImputer(strategy="median"))])

# 類別補眾數 + One-hot
cat_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_tf, num_cols),
    ("cat", cat_tf, cat_cols)
])

rf_sel = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])

rf_sel.fit(X, y)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['MSSubClass', 'LotFrontage',
                                                   'LotArea', 'OverallQual',
                                                   'OverallCond', 'YearBuilt',
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                                   '1stFlrSF', '2ndFlrSF',
                                                   'LowQualFinSF', 'GrLivArea',
                                                   'BsmtFullBath',
                                                   'B...
                                                   'Neighborhood', 'Condition1',
                                                   'Condition2', 'BldgType',
                                                   'HouseStyle', 'RoofStyle',
                                                   'RoofMatl', 'Exterior1st',
                                                   'Exterior2nd', 'MasVnrType',
                                                   'ExterQual', 'ExterCond',
                                                   'Foundation', 'BsmtQual',
                                                   'BsmtCond', 'BsmtExposure',
                                                   'BsmtFinType1',
                                                   'BsmtFinType2', 'Heating',
                                                   'HeatingQC', 'CentralAir',
                                                   'Electrical', ...])])),
                ('model',
                 RandomForestRegressor(n_estimators=300, n_jobs=-1,
                                       random_state=42))])

In [41]:
# Cell 9: 計算原始特徵重要性 (包含 TotalQualityIndex)

# 取得 one-hot 後特徵名
feat_names = rf_sel.named_steps["prep"].get_feature_names_out()
importances = rf_sel.named_steps["model"].feature_importances_

df_imp = pd.DataFrame({
    "feat": feat_names,
    "imp": importances
})

# 還原原始欄位
def revert(name):
    if name.startswith("num__"):
        # 特殊處理新增的 TotalQualityIndex，它可能沒有底線
        return name.replace("num__", "")
    if name.startswith("cat__"):
        return name.replace("cat__", "").split("_")[0]
    return name

df_imp["orig"] = df_imp["feat"].apply(revert)

# 合併 one-hot 的原始欄位 importance（取 max）
imp_orig = df_imp.groupby("orig")["imp"].max().sort_values(ascending=False)

print("特徵重要性前 20 名 (含 TotalQualityIndex):")
imp_orig.head(20)

特徵重要性前 20 名 (含 TotalQualityIndex):


,imp
orig,
TotalQualityIndex,0.787775
GrLivArea,0.031185
2ndFlrSF,0.028619
BsmtFinSF1,0.016514
LotArea,0.013654
TotalBsmtSF,0.008747
1stFlrSF,0.007203
YearRemodAdd,0.006415
LotFrontage,0.005598


In [42]:
# Cell 10: TopK 特徵數量自動搜索

k_list = [10, 20, 30, 40, 50, 60]
results = []

for k in k_list:
    topk = imp_orig.head(k).index.tolist()

    # 訓練資料集使用最新的 X
    df_k = X[topk].copy()
    df_k["SalePrice"] = y

    # 固定 split
    tr, va = split_fixed(df_k)

    tr_tfdf = tfdf.keras.pd_dataframe_to_tf_dataset(tr, label="SalePrice", task=tfdf.keras.Task.REGRESSION)
    va_tfdf = tfdf.keras.pd_dataframe_to_tf_dataset(va, label="SalePrice", task=tfdf.keras.Task.REGRESSION)

    model = tfdf.keras.RandomForestModel(
        task=tfdf.keras.Task.REGRESSION,
        num_trees=400,
        max_depth=16,
        growing_strategy="BEST_FIRST_GLOBAL",
        random_seed=42
    )
    model.compile(metrics=["mse"])
    model.fit(tr_tfdf)

    eva = model.evaluate(va_tfdf, return_dict=True, verbose=0)
    rmse = eva["mse"] ** 0.5

    print(f"Top{k} RMSE = {rmse:,.2f}")
    results.append((k, rmse))

Use /tmp/tmp7af3v73s as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.345341. Found 1142 examples.
Training model...
Model trained in 0:00:00.612371
Compiling model...
Model compiled.
Top10 RMSE = 26,785.53
Use /tmp/tmpypc0707a as temporary training directory
Reading training dataset...
Training dataset read in 0:00:01.348537. Found 1142 examples.
Training model...
Model trained in 0:00:01.256122
Compiling model...
Model compiled.
Top20 RMSE = 26,944.33
Use /tmp/tmp1howur75 as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.899008. Found 1142 examples.
Training model...
Model trained in 0:00:01.406950
Compiling model...
Model compiled.
Top30 RMSE = 26,812.25
Use /tmp/tmpb3zfq8cb as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.960052. Found 1142 examples.
Training model...
Model trained in 0:00:01.472360
Compiling model...
Model compiled.
Top40 RMSE = 26,324

In [43]:
# Cell 11: 最佳 TopK 結果

best_k, best_rmse = sorted(results, key=lambda x: x[1])[0]
print("最佳 TopK =", best_k, "RMSE =", best_rmse)

最佳 TopK = 40 RMSE = 26324.346145726013


In [44]:
# Cell 12: 最終 RF 模型訓練 (使用最佳 TopK)

best_features = imp_orig.head(best_k).index.tolist()

df_final = X[best_features].copy()
df_final["SalePrice"] = y
tr, va = split_fixed(df_final)

tr_tfdf = tfdf.keras.pd_dataframe_to_tf_dataset(tr, label="SalePrice", task=tfdf.keras.Task.REGRESSION)
va_tfdf = tfdf.keras.pd_dataframe_to_tf_dataset(va, label="SalePrice", task=tfdf.keras.Task.REGRESSION)

final_model = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=800,
    max_depth=18,
    growing_strategy="BEST_FIRST_GLOBAL",
    random_seed=999
)
final_model.compile(metrics=["mse"])
final_model.fit(tr_tfdf)

eva = final_model.evaluate(va_tfdf, return_dict=True, verbose=0)
rmse_final = eva["mse"] ** 0.5

print("\n🎯 最終 Final Model RMSE =", f"{rmse_final:,.2f}")

Use /tmp/tmpb7yb4tj9 as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.440193. Found 1142 examples.
Training model...
Model trained in 0:00:01.380619
Compiling model...
Model compiled.

🎯 最終 Final Model RMSE = 26,265.02


In [45]:
# Cell 13: 固定一組 Top 50 特徵列表

fixed_k = 50
fixed_features = imp_orig.head(fixed_k).index.tolist()

print("使用 fixed TopK =", fixed_k)
print("特徵數量 =", len(fixed_features))
print(fixed_features)


使用 fixed TopK = 50
特徵數量 = 50
['TotalQualityIndex', 'GrLivArea', '2ndFlrSF', 'BsmtFinSF1', 'LotArea', 'TotalBsmtSF', '1stFlrSF', 'YearRemodAdd', 'LotFrontage', 'OpenPorchSF', 'BsmtUnfSF', 'GarageArea', 'YearBuilt', 'WoodDeckSF', 'GarageYrBlt', 'MSSubClass', 'OverallCond', 'MasVnrArea', 'TotRmsAbvGrd', 'MoSold', 'MSZoning', 'Fireplaces', 'GarageCars', 'BedroomAbvGr', 'FullBath', 'BsmtExposure', 'GarageFinish', 'YrSold', 'Neighborhood', 'BsmtFullBath', 'ScreenPorch', 'KitchenQual', 'BsmtQual', 'Exterior1st', 'EnclosedPorch', 'LotShape', 'SaleType', 'LandContour', 'Condition1', 'Exterior2nd', 'SaleCondition', 'PoolQC', 'BsmtFinType1', 'BsmtFinSF2', 'LotConfig', 'GarageType', 'CentralAir', 'Functional', 'HalfBath', 'LandSlope']


In [46]:
# Cell 14: 訓練 RF50 模型 (使用固定 Top 50 特徵)

# 建 Final dataset（用固定 Top50 特徵 + SalePrice）
dataset_rf50 = X[fixed_features].copy()
dataset_rf50["SalePrice"] = y

# 固定 split
train_rf50, valid_rf50 = split_fixed(dataset_rf50, test_ratio=0.2, seed=42)

train_rf50_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_rf50, label="SalePrice", task=tfdf.keras.Task.REGRESSION
)
valid_rf50_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    valid_rf50, label="SalePrice", task=tfdf.keras.Task.REGRESSION
)

# ⚙️ RF 設定
rf50_model = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=600,
    max_depth=16,
    growing_strategy="BEST_FIRST_GLOBAL",
    random_seed=42,
)
rf50_model.compile(metrics=["mse"])

print("開始訓練 RF50 模型 ...")
rf50_model.fit(train_rf50_ds)

eval_rf50 = rf50_model.evaluate(valid_rf50_ds, return_dict=True, verbose=0)
rmse_rf50 = eval_rf50["mse"] ** 0.5

print("\n📊 RF50 Model RMSE =", f"{rmse_rf50:,.2f}")

Use /tmp/tmp4nrzo_m9 as temporary training directory
開始訓練 RF50 模型 ...
Reading training dataset...
Training dataset read in 0:00:00.523481. Found 1142 examples.
Training model...
Model trained in 0:00:01.180957
Compiling model...
Model compiled.

📊 RF50 Model RMSE = 26,425.04


In [47]:
# Cell 15: 訓練 GBT 模型（Top50 特徵）- 通常效果更好

gbt_model = tfdf.keras.GradientBoostedTreesModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=600,          # 樹比較多，慢一點但通常比較準
    max_depth=8,            # 不要太深，避免 overfit
    shrinkage=0.05,         # learning rate 類似參數
    subsample=0.8,          # 每輪用 80% 資料做 boosting
    growing_strategy="BEST_FIRST_GLOBAL",
    random_seed=42,
)

gbt_model.compile(metrics=["mse"])

print("開始訓練 GBT 模型（Top50）...")
gbt_model.fit(train_rf50_ds)

eval_gbt = gbt_model.evaluate(valid_rf50_ds, return_dict=True, verbose=0)
rmse_gbt = eval_gbt["mse"] ** 0.5

print("\n🎯 GBT Model RMSE =", f"{rmse_gbt:,.2f}")


Use /tmp/tmpx70nj8p9 as temporary training directory
開始訓練 GBT 模型（Top50）...
Reading training dataset...
Training dataset read in 0:00:00.680967. Found 1142 examples.
Training model...
Model trained in 0:00:02.328683
Compiling model...
Model compiled.

🎯 GBT Model RMSE = 23,907.00


In [48]:
# Cell 16: 最終預測與提交 (使用 GBT 模型)

# 使用 fixed_features (Top50 特徵，包含新特徵)
test_final = test[fixed_features]
test_tfdf = tfdf.keras.pd_dataframe_to_tf_dataset(test_final, task=tfdf.keras.Task.REGRESSION)

# 使用表現最佳的 GBT 模型進行預測
preds = gbt_model.predict(test_tfdf).squeeze()

# 建立提交檔
sub = pd.DataFrame({"Id": test_ids, "SalePrice": preds})
# 更改檔名以便區分這是 GBT 模型的結果
sub.to_csv("submission_gbt_top51.csv", index=False)

print("✅ 已使用 GBT 模型預測並生成 submission_gbt_top51.csv")
sub.head()

2/2 [==============================] - 0s 10ms/step
✅ 已使用 GBT 模型預測並生成 submission_gbt_top51.csv


,Id,SalePrice
0,1461,121638.468750
1,1462,151630.718750
2,1463,177993.406250
3,1464,181055.453125
4,1465,196456.968750
